In [ ]:
!wget --quiet https://raw.githubusercontent.com/tensorflow/models/master/official/nlp/tools/tokenization.py

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn import preprocessing
import tensorflow as tf
import tensorflow_hub as hub
import torch

import tokenization
from bert import tokenization
import gc
from keras.utils import to_categorical

from tensorflow.keras.models import Model
from tensorflow.keras.models import load_model
from sklearn.metrics import confusion_matrix , classification_report, ConfusionMatrixDisplay
from sklearn.utils import shuffle

import matplotlib.pyplot as plt
import re
import seaborn as sns
gc.collect()

In [ ]:
print(tf.__version__)

In [ ]:
# tf.config.set_visible_devices([], 'GPU')
physical_devices = tf.config.list_physical_devices('GPU')
tf.config.experimental.set_memory_growth(physical_devices[0], True)

In [ ]:
original_dataset = pd.read_csv('Data Set/Huge dataset/HugeDatasetWhole_labelsFakeReal.csv', nrows=1000000)

In [ ]:
fake_data = original_dataset[original_dataset['label'] == 'Fake']
real_data = original_dataset[original_dataset['label'] == 'Real']

num_fake_samples = 4500
num_real_samples = 5500

fake_samples = fake_data.sample(n=num_fake_samples, random_state=56)
real_samples = real_data.sample(n=num_real_samples, random_state=56)

df = pd.concat([fake_samples, real_samples], ignore_index=True)
df = shuffle(df, random_state=56)
df = df.reset_index(drop=True)
df.text = df.text.astype(str)

class_distribution = df['label'].value_counts(normalize=True)
print("New Dataset Class Distribution:")
print(class_distribution)

In [ ]:
#NEW: 1 Fake, 0 real
df['label'] = df['label'].replace({'Fake' : 0, 'Real' : 1})

In [ ]:
X = df['text']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state = 56)

In [ ]:
bert_url = 'https://tfhub.dev/tensorflow/bert_en_uncased_L-12_H-768_A-12/4'
bert = hub.KerasLayer(bert_url, trainable=True)

In [ ]:
import sys
from absl import flags
sys.argv=['preserve_unused_tokens=False']
flags.FLAGS(sys.argv)

In [ ]:
vocab = bert.resolved_object.vocab_file.asset_path.numpy()
lower_case = bert.resolved_object.do_lower_case.numpy()
tokenizer = tokenization.FullTokenizer(vocab, lower_case)

def bert_encode(texts, tokenizer, max_len=512):
    all_tokens = []
    all_masks = []
    all_segments = []
    
    for text in texts:
        text = tokenizer.tokenize(text)
        
        text = text[:max_len-2]
        input_sequence = ["[CLS]"] + text + ["[SEP]"]
        pad_len = max_len-len(input_sequence)
        
        tokens = tokenizer.convert_tokens_to_ids(input_sequence) + [0] * pad_len
        pad_masks = [1] * len(input_sequence) + [0] * pad_len
        segment_ids = [0] * max_len
        
        all_tokens.append(tokens)
        all_masks.append(pad_masks)
        all_segments.append(segment_ids)
        
    return np.array(all_tokens), np.array(all_masks), np.array(all_segments)

In [ ]:
def build_model(bert_layer, max_len=512):
    input_word_ids = tf.keras.Input(shape=(max_len,), dtype=tf.int32, name="input_word_ids")
    input_mask = tf.keras.Input(shape=(max_len,), dtype=tf.int32, name="input_mask")
    segment_ids = tf.keras.Input(shape=(max_len,), dtype=tf.int32, name="segment_ids")

    outputs = bert_layer({'input_word_ids': input_word_ids,
                          'input_mask': input_mask,
                          'input_type_ids': segment_ids})

    sequence_output = outputs['sequence_output']
    pooled_output = outputs['pooled_output']
    
    clf_output = sequence_output[:, 0, :]
    
    lay = tf.keras.layers.Dense(64, activation='relu')(clf_output)
    lay = tf.keras.layers.Dropout(0.6)(lay)
    lay = tf.keras.layers.Dense(32, activation='relu')(lay)
    lay = tf.keras.layers.Dropout(0.55)(lay)
    out = tf.keras.layers.Dense(1, activation='sigmoid')(lay)
    
    bert_model = tf.keras.models.Model(inputs=[input_word_ids, input_mask, segment_ids], outputs=out)
    bert_model.compile(tf.keras.optimizers.Adam(learning_rate=1e-5), loss='binary_crossentropy', metrics=['accuracy'])
    
    return bert_model

In [ ]:
max_len = 250
train_text = bert_encode(X_train.values, tokenizer, max_len=max_len)
test_text = bert_encode(X_test.values, tokenizer, max_len=max_len)

In [ ]:
bert_model = build_model(bert, max_len=max_len)
bert_model.summary()

In [ ]:
checkpoint = tf.keras.callbacks.ModelCheckpoint('Data Set/Huge dataset/Bert/HugeDataset_BERT_model_checkpoint_test1.h5', monitor='val_loss', save_best_only=True, verbose=1)
earlystopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1)

train_sh = bert_model.fit(
    train_text, y_train,
    validation_split=0.2,
    epochs=15,
    callbacks=[checkpoint, earlystopping],
    batch_size=2,
    verbose=1
)

In [ ]:
bert_model.save(f'Data Set/Huge dataset/Bert/HugeDataset_BERT_model_test1.h5')
bert_model.save_weights(f'Data Set/Huge dataset/Bert/HugeDataset_BERT_model_weights_test1.h5')

In [ ]:
loss, accuracy = bert_model.evaluate(x=test_text, y=y_test, verbose=1)

print("\nTest Loss:", loss)
print("\nTest Accuracy:", accuracy)

In [ ]:
fake_data = original_dataset[original_dataset['label'] == 'Fake']
real_data = original_dataset[original_dataset['label'] == 'Real']

num_fake_samples = 45000
num_real_samples = 55000

fake_samples = fake_data.sample(n=num_fake_samples, random_state=56)
real_samples = real_data.sample(n=num_real_samples, random_state=56)

news_df = pd.concat([fake_samples, real_samples], ignore_index=True)
news_df = shuffle(news_df, random_state=56)
news_df = news_df.reset_index(drop=True)

huge_whole_text = news_df['text']
huge_whole_text = bert_encode(huge_whole_text.values, tokenizer, max_len=max_len)
predictions = bert_model.predict(huge_whole_text)

In [ ]:
#NEW: 1 Fake, 0 real
print(predictions)
predicted_classes = (predictions > 0.5).astype('int32')

predicted_labels = []
for predicted_class in predicted_classes:
    if predicted_class == 1:
        predicted_labels.append("Real")
    elif predicted_class == 0:
        predicted_labels.append("Fake")
    else:
        predicted_labels.append("ERROR/UNKNOWN")

true_labels = news_df['label'].replace({1:'Real', 0:'Fake'})
print(classification_report(true_labels, predicted_labels))

In [ ]:
print(train_sh.history['loss'])
print(train_sh.history['val_loss'])
print(train_sh.history['accuracy'])
print(train_sh.history['val_accuracy'])

plt.figure(1)
plt.plot(train_sh.history['loss'])
plt.plot(train_sh.history['val_loss'])
plt.legend(['training', 'validation'])
plt.title('Loss')
plt.xlabel('epoch')


plt.figure(2)
plt.plot(train_sh.history['accuracy'])
plt.plot(train_sh.history['val_accuracy'])
plt.legend(['training', 'validation'])
plt.title('Accuracy')
plt.xlabel('epoch')

plt.show()

In [ ]:
#NEW: 1 Fake, 0 real
original_dataset2 = pd.read_csv('Data Set/Huge dataset/HugeDatasetWhole_labelsFakeReal.csv', nrows=1000000)
fake_data = original_dataset[original_dataset['label'] == 'Fake']
real_data = original_dataset[original_dataset['label'] == 'Real']

num_fake_samples = 45000
num_real_samples = 55000

fake_samples = fake_data.sample(n=num_fake_samples, random_state=56)
real_samples = real_data.sample(n=num_real_samples, random_state=56)

news_df = pd.concat([fake_samples, real_samples], ignore_index=True)
news_df = shuffle(news_df, random_state=56)
news_df = news_df.reset_index(drop=True)

X_test_huge = news_df['text']
X_test_huge = bert_encode(X_test_huge.values, tokenizer, max_len=max_len)

predictions_huge = bert_model.predict(X_test_huge)

#NEW: 1 Fake, 0 real
print(predictions_huge)
predicted_classes_huge = (predictions_huge > 0.5).astype('int32')

predicted_labels_huge = []
for predicted_class in predicted_classes_huge:
    if predicted_class == 1:
        predicted_labels_huge.append("Real")
    elif predicted_class == 0:
        predicted_labels_huge.append("Fake")
    else:
        predicted_classes_huge.append("ERROR/UNKNOWN")

true_labels_huge = news_df['label']
print(classification_report(true_labels_huge, predicted_labels_huge))

In [ ]:
liar_dataset_test = pd.read_csv('Data Set/LIAR dataset/liar_prepro_test_df_nolemmaNstopwords.csv')
liar_dataset_train = pd.read_csv('Data Set/LIAR dataset/liar_prepro_train_df_nolemmaNstopwords.csv')
liar_dataset_valid = pd.read_csv('Data Set/LIAR dataset/liar_prepro_valid_df_nolemmaNstopwords.csv')
liar_dataset = pd.concat([liar_dataset_test, liar_dataset_train, liar_dataset_valid], ignore_index=True)

X_test_liar = liar_dataset['text']
X_test_liar = bert_encode(X_test_liar.values, tokenizer, max_len=max_len)

predictions_liar = bert_model.predict(X_test_liar)

#NEW: 1 Fake, 0 real
print(predictions_liar)
predicted_classes_liar = (predictions_liar > 0.5).astype('int32')

predicted_labels_liar = []
for predicted_class in predicted_classes_liar:
    if predicted_class == 1:
        predicted_labels_liar.append("Real")
    elif predicted_class == 0:
        predicted_labels_liar.append("Fake")
    else:
        predicted_labels_liar.append("ERROR/UNKNOWN")

true_labels_liar = liar_dataset['label']
print(classification_report(true_labels_liar, predicted_labels_liar))

In [ ]:
#NEW: 1 Fake, 0 real
welfake_dataset = pd.read_csv('Data Set/WELFake Dataset/new_WELFake_prepro_df_nolemmaNstopwordsNreuters.csv')
welfake_dataset['text'].fillna('', inplace=True)
welfake_dataset['label'] = welfake_dataset['label'].replace({1: 'Fake', 0: 'Real'})

X_test_welfake = welfake_dataset['text']
X_test_welfake = bert_encode(X_test_welfake.values, tokenizer, max_len=max_len)

predictions_welfake = bert_model.predict(X_test_welfake)

#NEW: 1 Fake, 0 real
print(predictions_welfake)
predicted_classes_welfake = (predictions_welfake > 0.5).astype('int32')

predicted_labels_welfake = []
for predicted_class in predicted_classes_welfake:
    if predicted_class == 1:
        predicted_labels_welfake.append("Real")
    elif predicted_class == 0:
        predicted_labels_welfake.append("Fake")
    else:
        predicted_labels_welfake.append("ERROR/UNKNOWN")

true_labels_welfake = welfake_dataset['label']
print(classification_report(true_labels_welfake, predicted_labels_welfake))